# Practical 10: LSTM Sentiment Classifier with PyTorch

**Concept:** A **Recurrent Neural Network (RNN)** processes text word-by-word and carries a hidden state that acts as a "memory" of what came before. A **Long Short-Term Memory (LSTM)** cell is an improved RNN unit with gates that control what to remember and what to forget — it learns **long-range dependencies** without suffering from the vanishing-gradient problem.

In this practical we build an LSTM sentiment classifier in **PyTorch** end-to-end:
1. Prepare the data (tokenization, vocabulary, **padding**).
2. Define the model architecture (`Embedding → LSTM → Linear`).
3. Train the model with a training loop.
4. Evaluate accuracy and test on new sentences.


## Install PyTorch


In [15]:
%pip install torch



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /home/tejas/.local/share/pipx/venvs/notebook/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Import Required Libraries


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np

import nltk
from nltk.tokenize import word_tokenize

from torch.utils.data import DataLoader, TensorDataset

# Fix the random seed so results are reproducible
torch.manual_seed(42)

print("PyTorch version:", torch.__version__)


PyTorch version: 2.13.0+cu130


## Download NLTK Tokenizer Data


In [17]:
nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to /home/tejas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/tejas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Create the Dataset
We generate a synthetic dataset of 60 movie reviews (30 positive, 30 negative) by filling positive/negative **sentence templates** with sentiment adjectives. Synthetic data lets us control the class balance while keeping the corpus small enough to train quickly. The training/test split is covered in the next section.


In [18]:
import random

# Sentiment words and sentence templates used to build the synthetic corpus
pos_adj = ["amazing", "wonderful", "fantastic", "brilliant", "exciting",
           "heartwarming", "stunning", "engaging", "superb", "delightful"]
neg_adj = ["boring", "terrible", "awful", "dull", "predictable",
           "disappointing", "confusing", "dreadful", "mediocre", "tedious"]

pos_templates = [
    "the movie was {a} and {b}",
    "a {a} story with {b} performances",
    "i loved every moment of this {a} film",
    "an {a} visual experience, highly recommended",
    "great acting and a very {a} plot",
    "i enjoyed it a lot, definitely worth watching",
    "the ending was {a} and {b}",
    "{a} performances from the entire cast",
    "the direction was {a} and the story was {b}",
    "a truly {a} film that i would watch again",
]

neg_templates = [
    "the movie was {a} and {b}",
    "a {a} story with {b} acting",
    "i hated this film, complete waste of time",
    "the plot made no sense at all",
    "a dull and {a} experience",
    "worst movie i have ever seen",
    "the acting was {a} and the dialogue felt fake",
    "an utterly {a} film, avoid it",
    "the story dragged on and felt {a}",
    "boring and predictable, do not waste your time",
]

# Generate 3 reviews per template (30 positive, 30 negative)
random.seed(42)
reviews = []

for template in pos_templates:
    for _ in range(3):
        a, b = random.choice(pos_adj), random.choice(pos_adj)
        reviews.append((template.format(a=a, b=b), 1))

for template in neg_templates:
    for _ in range(3):
        a, b = random.choice(neg_adj), random.choice(neg_adj)
        reviews.append((template.format(a=a, b=b), 0))

# Shuffle so positive/negative samples are mixed
random.shuffle(reviews)

print("Dataset")
print("-" * 50)
n_pos = sum(1 for _, label in reviews if label == 1)
n_neg = sum(1 for _, label in reviews if label == 0)
print(f"Total samples: {len(reviews)}  (Positive: {n_pos}  Negative: {n_neg})")
print()
for text, label in reviews[:8]:
    print(f"[{label}] {text}")


Dataset
--------------------------------------------------
Total samples: 60  (Positive: 30  Negative: 30)

[0] a dull and dull experience
[0] boring and predictable, do not waste your time
[1] the movie was brilliant and fantastic
[0] the movie was dull and terrible
[1] i loved every moment of this superb film
[1] great acting and a very exciting plot
[1] an amazing visual experience, highly recommended
[1] the movie was exciting and brilliant


## 1. Tokenization & Vocabulary
We tokenize each review and build a word → index mapping. We reserve `0` for the padding token and `1` for unknown words.


In [19]:
# Tokenize every review
tokenized = [word_tokenize(text) for text, _ in reviews]

# Build the vocabulary from all tokens
vocab = set(w for tokens in tokenized for w in tokens)

word2idx = {"<pad>": 0, "<unk>": 1}
for word in sorted(vocab):
    word2idx[word] = len(word2idx)

print(f"Vocabulary size: {len(word2idx)}")
print("First 15 entries:", list(word2idx.items())[:15])


Vocabulary size: 86
First 15 entries: [('<pad>', 0), ('<unk>', 1), (',', 2), ('a', 3), ('acting', 4), ('again', 5), ('all', 6), ('amazing', 7), ('an', 8), ('and', 9), ('at', 10), ('avoid', 11), ('awful', 12), ('boring', 13), ('brilliant', 14)]


## 2. Encoding & Padding
Neural networks need fixed-length inputs, so we convert each review to a list of indices and **pad** (or truncate) it to `MAX_LEN` tokens using the padding index `0`.


In [20]:
MAX_LEN = 12


def encode_and_pad(tokens):
    """Convert tokens to indices and pad/truncate to MAX_LEN."""
    ids = [word2idx.get(w, word2idx["<unk>"]) for w in tokens]
    if len(ids) > MAX_LEN:
        ids = ids[:MAX_LEN]
    else:
        ids = ids + [word2idx["<pad>"]] * (MAX_LEN - len(ids))
    return ids


X = np.array([encode_and_pad(tokens) for tokens in tokenized], dtype=np.int64)
y = np.array([label for _, label in reviews], dtype=np.int64)

print("Padded index sequences")
print("-" * 50)
for i in range(3):
    print(f"len={len(X[i])}  {X[i]}")

print(f"\nX shape: {X.shape}  (samples, sequence length)")


Padded index sequences
--------------------------------------------------
len=12  [ 3 26  9 26 34  0  0  0  0  0  0  0]
len=12  [13  9 59  2 23 54 77 85 71  0  0  0]
len=12  [69 52 76 14  9 36  0  0  0  0  0  0]

X shape: (60, 12)  (samples, sequence length)


## 3. Train/Test Split & DataLoader
We convert to PyTorch tensors, split off a test set and wrap the data in a `DataLoader` for batched training.


In [21]:
X_tensor = torch.tensor(X)
y_tensor = torch.tensor(y)

# Split (keep the same random state so it is reproducible)
test_size = 15  # 25% of the data
X_train = X_tensor[:-test_size]
y_train = y_tensor[:-test_size]
X_test = X_tensor[-test_size:]
y_test = y_tensor[-test_size:]

train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
test_loader = DataLoader(test_data, batch_size=8)

print("Train samples:", len(train_data), "| Test samples:", len(test_data))
print("Train batches:", len(train_loader), "| Test batches:", len(test_loader))


Train samples: 45 | Test samples: 15
Train batches: 6 | Test batches: 2


## 4. Model Architecture
The model is a small LSTM classifier:
- **Embedding layer** maps each word index to a dense vector.
- **LSTM layer** processes the sequence and produces hidden states.
- **Linear layer** classifies the last hidden state into 2 sentiment classes.


In [22]:
class LSTMSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x shape: (batch, seq_len)
        embedded = self.embedding(x)                 # (batch, seq_len, emb_dim)
        output, (hidden, cell) = self.lstm(embedded) # hidden: (layers, batch, hidden_dim)
        last_hidden = hidden[-1]                     # use the final hidden state
        out = self.fc(self.dropout(last_hidden))     # (batch, output_dim)
        return out


VOCAB_SIZE = len(word2idx)
EMBED_DIM = 32
HIDDEN_DIM = 64
OUTPUT_DIM = 2

model = LSTMSentimentClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM)

print(model)


LSTMSentimentClassifier(
  (embedding): Embedding(86, 32, padding_idx=0)
  (lstm): LSTM(32, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


## 5. Training Setup
Cross-entropy loss suits classification; Adam is a reliable optimizer. We also compute accuracy at the end of every epoch.


In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 40


def accuracy_fn(logits, labels):
    predictions = torch.argmax(logits, dim=1)
    return (predictions == labels).float().mean().item()


## 6. Training Loop
For each epoch: forward pass → compute loss → backpropagate → update weights. Loss should drop as training proceeds.


In [24]:
train_losses = []
train_accs = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_acc = 0.0

    for inputs, labels in train_loader:
        optimizer.zero_grad()

        logits = model(inputs)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * inputs.size(0)
        epoch_acc += accuracy_fn(logits, labels) * inputs.size(0)

    avg_loss = epoch_loss / len(train_data)
    avg_acc = epoch_acc / len(train_data)
    train_losses.append(avg_loss)
    train_accs.append(avg_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>2}  |  Loss: {avg_loss:.4f}  |  Train Acc: {avg_acc:.4f}")


Epoch  1  |  Loss: 0.6917  |  Train Acc: 0.5111
Epoch  5  |  Loss: 0.6663  |  Train Acc: 0.7333
Epoch 10  |  Loss: 0.1734  |  Train Acc: 0.9778
Epoch 15  |  Loss: 0.0036  |  Train Acc: 1.0000
Epoch 20  |  Loss: 0.0018  |  Train Acc: 1.0000
Epoch 25  |  Loss: 0.0011  |  Train Acc: 1.0000
Epoch 30  |  Loss: 0.0012  |  Train Acc: 1.0000
Epoch 35  |  Loss: 0.0008  |  Train Acc: 1.0000
Epoch 40  |  Loss: 0.0007  |  Train Acc: 1.0000


## 7. Evaluation on the Test Set
Now we evaluate the model on reviews it has never seen, with the model in `eval()` mode and gradients disabled.


In [25]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            logits = model(inputs)
            predictions = torch.argmax(logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total


test_acc = evaluate(model, test_loader)

print("Test Evaluation")
print("-" * 50)
print(f"Test Accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")

# Show per-sample predictions
model.eval()
with torch.no_grad():
    logits = model(X_test)
    preds = torch.argmax(logits, dim=1)

print()
print("True vs Predicted")
print("-" * 50)
for i, (true_label, pred_label) in enumerate(zip(y_test, preds)):
    mark = "[OK]" if true_label == pred_label else "[X]"
    review_text = reviews[len(train_data) + i][0]
    print(f"Review: {review_text}")
    print(f"  True: {true_label.item()}  Predicted: {pred_label.item()}  {mark}")


Test Evaluation
--------------------------------------------------
Test Accuracy: 0.8000 (80.00%)

True vs Predicted
--------------------------------------------------
Review: i enjoyed it a lot, definitely worth watching
  True: 1  Predicted: 1  [OK]
Review: a disappointing story with dull acting
  True: 0  Predicted: 0  [OK]
Review: the acting was dull and the dialogue felt fake
  True: 0  Predicted: 0  [OK]
Review: i loved every moment of this brilliant film
  True: 1  Predicted: 1  [OK]
Review: worst movie i have ever seen
  True: 0  Predicted: 0  [OK]
Review: a dull and disappointing experience
  True: 0  Predicted: 0  [OK]
Review: the movie was wonderful and amazing
  True: 1  Predicted: 0  [X]
Review: a predictable story with terrible acting
  True: 0  Predicted: 1  [X]
Review: i enjoyed it a lot, definitely worth watching
  True: 1  Predicted: 1  [OK]
Review: a awful story with disappointing acting
  True: 0  Predicted: 1  [X]
Review: a truly exciting film that i would watch ag

## 8. Predicting on New Sentences
Finally we predict sentiment for brand-new reviews by tokenizing, encoding, padding and running a forward pass.


In [26]:
def predict_sentiment(model, sentence):
    tokens = word_tokenize(sentence)
    ids = encode_and_pad(tokens)
    tensor = torch.tensor([ids])

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        prob = torch.softmax(logits, dim=1)
        predicted = torch.argmax(prob, dim=1).item()

    sentiment = "Positive" if predicted == 1 else "Negative"
    confidence = prob[0, predicted].item()
    return sentiment, confidence


new_reviews = [
    "a truly wonderful and heartwarming film",
    "the movie was slow and utterly boring",
    "absolutely loved this brilliant masterpiece",
]

print("Predictions on New Reviews")
print("-" * 50)
for review in new_reviews:
    sentiment, confidence = predict_sentiment(model, review)
    print(f"Review: {review}")
    print(f"  Predicted: {sentiment}  (confidence: {confidence:.3f})")
    print()


Predictions on New Reviews
--------------------------------------------------
Review: a truly wonderful and heartwarming film
  Predicted: Positive  (confidence: 0.999)

Review: the movie was slow and utterly boring
  Predicted: Negative  (confidence: 0.999)

Review: absolutely loved this brilliant masterpiece
  Predicted: Positive  (confidence: 1.000)



## Summary

We built a complete LSTM sentiment classifier in PyTorch:
1. **Data preparation** — tokenized reviews, built a vocabulary and **padded** sequences to a fixed length.
2. **Model architecture** — `Embedding → LSTM → Linear` defined as an `nn.Module` class.
3. **Training loop** — forward/backward passes with `CrossEntropyLoss` and `Adam` over multiple epochs.
4. **Evaluation** — accuracy on a held-out test set and predictions on new sentences.

LSTMs were the state of the art before Transformers; the same padding + embedding + recurrent structure generalizes to tasks like POS tagging, machine translation and speech recognition.
